In [ ]:
import os
os.chdir("..")

In [ ]:
from prfood_repl import FoodDeseart
import hashlib
import tempfile
from jp_tools import download
import geopandas as gpd
from pathlib import Path
from CensusForge import CensusAPI

import polars as pl

import os
from dotenv import load_dotenv

load_dotenv()


fd = FoodDeseart()
capi = CensusAPI(os.getenv("CENSUS_TOKEN"))


In [ ]:
capi.url

In [ ]:
capi.query(
    dataset="acs-acs5-profile",
    params_list=[
                "DP03_0001E",
                "DP03_0003E",
                "DP03_0004E",
                "DP03_0005E",
                "DP03_0006E",
                "DP03_0007E",
                "DP03_0014E",
                "DP03_0025E",
                "DP03_0033E",
                "DP03_0051E",
                "DP03_0052E",
                "DP03_0053E",
                "DP03_0054E",
                "DP03_0055E",
                "DP03_0056E",
                "DP03_0057E",
                "DP03_0058E",
                "DP03_0059E",
                "DP03_0060E",
                "DP03_0061E",
                        ],
    geography="zip code tabulation area",
    year="2019",
    skip_checks=True
)

In [ ]:
r = CensusAPI(os.getenv("CENSUS_TOKEN")).query(
                    params_list=[
                        "DP03_0001E",
                        "DP03_0003E",
                        "DP03_0004E",
                        "DP03_0005E",
                        "DP03_0006E",
                        "DP03_0007E",
                        "DP03_0014E",
                        "DP03_0025E",
                        "DP03_0033E",
                        "DP03_0051E",
                        "DP03_0052E",
                        "DP03_0053E",
                        "DP03_0054E",
                        "DP03_0055E",
                        "DP03_0056E",
                        "DP03_0057E",
                        "DP03_0058E",
                        "DP03_0059E",
                        "DP03_0060E",
                        "DP03_0061E",
                    ],
                    year=2019,
                    geography="zip code tabulation area",
                    dataset="acs-acs5-profile",
                    extra="&in=state:72",
                    skip_checks=True
                )
df = pl.DataFrame(r)
df = df.rename(df.row(0, named=True))
df = df.slice(1).with_columns(
    pl.col("*").exclude("state", "county").cast(pl.Float64)
)
df = df.rename(
    {
        "dp03_0001e": "total_population",
        "dp03_0003e": "total_civilian_force",
        "dp03_0004e": "total_labor_force",
        "dp03_0005e": "total_unemployed",
        "dp03_0006e": "total_armed_forces",
        "dp03_0007e": "total_not_labor",
        "dp03_0014e": "total_own_children",
        "dp03_0025e": "mean_travel_time",
        "dp03_0033e": "agr_fish_employment",
        "dp03_0051e": "total_house",
        "dp03_0052e": "inc_less_10k",
        "dp03_0053e": "inc_10k_15k",
        "dp03_0054e": "inc_15k_25k",
        "dp03_0055e": "inc_25k_35k",
        "dp03_0056e": "inc_35k_50k",
        "dp03_0057e": "inc_50k_75k",
        "dp03_0058e": "inc_75k_100k",
        "dp03_0059e": "inc_100k_150k",
        "dp03_0060e": "inc_150k_200k",
        "dp03_0061e": "inc_more_200k",
    }
)
df = df.with_columns(
    year=_year,
    geoid=pl.col("state").str.zfill(2) + pl.col("county").str.zfill(3),
)
df

In [ ]:
   def pull_dp03(self) -> pl.DataFrame:

        for _year in range(2012, 2020):
            if (
                self.conn.sql(f"SELECT * FROM 'DP03Table' WHERE year={_year}")
                .df()
                .empty
            ):
                try:
                    logging.info(f"pulling {_year} data")
                    df = self.pull_query(
                        params=[
                            "DP03_0001E",
                            "DP03_0003E",
                            "DP03_0004E",
                            "DP03_0005E",
                            "DP03_0006E",
                            "DP03_0007E",
                            "DP03_0014E",
                            "DP03_0025E",
                            "DP03_0033E",
                            "DP03_0051E",
                            "DP03_0052E",
                            "DP03_0053E",
                            "DP03_0054E",
                            "DP03_0055E",
                            "DP03_0056E",
                            "DP03_0057E",
                            "DP03_0058E",
                            "DP03_0059E",
                            "DP03_0060E",
                            "DP03_0061E",
                        ],
                        year=_year,
                    )


In [ ]:
    def pull_query(self, params: list, year: int) -> pl.DataFrame:
        # prepare custom census query
        param = ",".join(params)
        base = "https://api.census.gov/data/"
        flow = "/acs/acs5/profile"
        url = f"{base}{year}{flow}?get={param}&for=zip%20code%20tabulation%20area:*&in=state:72"
        logging.debug(url)
        df = pl.DataFrame(requests.get(url).json())

        # get names from DataFrame
        names = df.select(pl.col("column_0")).transpose()
        names = names.to_dicts().pop()
        names = dict((k, v.lower()) for k, v in names.items())

        # Pivot table
        df = df.drop("column_0").transpose()
        return df.rename(names).with_columns(year=pl.lit(year))